# Data Exploration: mtDNA Sequence Corpus

This notebook documents the structure and statistics of the raw sequence datasets
before they enter the preprocessing pipeline. It covers five questions:

1. **Haplogroup distribution** — how phylogenetically diverse is the corpus?
2. **Shannon entropy per genomic position** — which regions are most variable, and does the D-loop stand out as expected?
3. **N-content distribution** — quality filter threshold selection
4. **Sequence length distribution before normalization** — how much length variation exists in raw data?
5. **Geographic distribution** — population sampling bias in HmtDB

Run `mtdna-download --source hmtdb` and `mtdna-download --source ncbi-refseq` before executing this notebook.
If the data is not yet downloaded, the notebook falls back to a synthetic dataset for illustration.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

FIGURES_DIR = Path("../docs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

HMTDB_DIR = Path("../data/raw/hmtdb")
NCBI_DIR  = Path("../data/raw/ncbi")
RCRS_LENGTH = 16569

In [ ]:
# ---------------------------------------------------------------------------
# Load data (real or synthetic)
# ---------------------------------------------------------------------------

USE_SYNTHETIC = not (HMTDB_DIR / "sequences.fasta").exists()

if USE_SYNTHETIC:
    print("Real data not found — using synthetic data for illustration.")
    print("Run `mtdna-download --source hmtdb` to get the real dataset.")

    rng = np.random.default_rng(42)
    N_SYNTH = 500
    HAPLOGROUPS = [
        "H", "H", "H", "H", "H", "H",  # H dominates in European HmtDB
        "J", "J", "T", "T", "U", "U", "K",
        "L0", "L1", "L2", "L3", "L3", "L3",
        "M", "N", "R", "B", "C", "D", "HV",
    ]
    hg_labels = rng.choice(HAPLOGROUPS, size=N_SYNTH)
    # Synthetic sequences: mostly ACGT with higher variability in D-loop region
    seqs = []
    for _ in range(N_SYNTH):
        seq = rng.choice(list("ACGT"), size=RCRS_LENGTH)
        # D-loop (positions 576-16024) is more variable — increase substitution rate
        dloop_mask = np.zeros(RCRS_LENGTH, dtype=bool)
        dloop_mask[576:16024] = True
        # Outside D-loop: reduce variability by locking most positions to 'A'
        non_dloop = ~dloop_mask
        lock = rng.random(RCRS_LENGTH) < 0.85
        seq[non_dloop & lock] = ord('A')
        seqs.append("".join(chr(c) if isinstance(c, int) else c for c in seq))

    # Synthetic length variation (before normalization)
    lengths_raw = rng.integers(RCRS_LENGTH - 50, RCRS_LENGTH + 250, size=N_SYNTH)

    # Synthetic geographic data
    regions = ["Europe", "Africa", "Asia", "Americas", "Middle East", "South Asia"]
    region_probs = [0.55, 0.20, 0.12, 0.06, 0.04, 0.03]
    geo = rng.choice(regions, size=N_SYNTH, p=region_probs)

    df = pd.DataFrame({
        "accession": [f"SYNTH{i:05d}" for i in range(N_SYNTH)],
        "sequence": seqs,
        "haplogroup": hg_labels,
        "species": "homo_sapiens",
        "geographic_origin": geo,
        "length_raw": lengths_raw,
    })
else:
    from Bio import SeqIO
    import pyarrow.parquet as pq

    fasta_path = HMTDB_DIR / "sequences.fasta"
    meta_path  = HMTDB_DIR / "metadata.parquet"

    rows = [
        {"accession": r.id, "sequence": str(r.seq), "length_raw": len(r.seq)}
        for r in SeqIO.parse(fasta_path, "fasta")
    ]
    df = pd.DataFrame(rows)

    if meta_path.exists():
        meta = pd.read_parquet(meta_path)
        merge_cols = [c for c in ("haplogroup", "geographic_origin", "species") if c in meta.columns]
        if merge_cols:
            id_col = "accession" if "accession" in meta.columns else meta.columns[0]
            meta = meta.rename(columns={id_col: "accession"})
            df = df.merge(meta[["accession"] + merge_cols], on="accession", how="left")

    for col in ("haplogroup", "geographic_origin", "species"):
        if col not in df.columns:
            df[col] = None

    print(f"Loaded {len(df):,} sequences from HmtDB")

print(df.shape)
df.head(3)

## 1. Haplogroup Distribution

H is the most common haplogroup in HmtDB because the database has a European sampling bias.
L3 is the phylogenetic root of all non-African haplogroups. Seeing L0/L1/L2/L3 in the corpus
is important for the model to learn deep evolutionary structure.

In [ ]:
hg_counts = (
    df["haplogroup"].dropna()
    .str.upper()
    .str.extract(r"^([A-Z][0-9]?)", expand=False)  # top-level haplogroup letter
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(hg_counts.index, hg_counts.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Haplogroup (top-level)")
ax.set_ylabel("Number of sequences")
ax.set_title("Haplogroup distribution in the training corpus")
ax.bar_label(bars, fmt="%d", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "haplogroup_distribution.png", bbox_inches="tight")
plt.show()

print(f"H fraction: {hg_counts.get('H', 0) / hg_counts.sum():.1%}")
print(f"L-root fraction (L0+L1+L2): "
      f"{(hg_counts.get('L0',0)+hg_counts.get('L1',0)+hg_counts.get('L2',0))/hg_counts.sum():.1%}")

## 2. Shannon Entropy Per Genomic Position

Shannon entropy measures per-position nucleotide diversity across all sequences.
The D-loop (positions 576–16024) is the non-coding control region and evolves much faster
than protein-coding, tRNA, or rRNA genes. We expect ~7x higher entropy in the D-loop.

This asymmetry is one reason circular positional encoding matters: the model needs to
correctly assign positions to the right side of the D-loop boundary.

In [ ]:
DLOOP_START = 576
DLOOP_END   = 16024  # approximate; precise boundaries vary by annotation

# Compute per-position nucleotide frequency matrix
# Subsample for speed if corpus is large
sample_seqs = df["sequence"].dropna().tolist()
if len(sample_seqs) > 2000:
    rng_sample = np.random.default_rng(42)
    idx = rng_sample.choice(len(sample_seqs), size=2000, replace=False)
    sample_seqs = [sample_seqs[i] for i in idx]

# Truncate or pad each sequence to RCRS_LENGTH for the matrix
seq_matrix = np.array(
    [list(s[:RCRS_LENGTH].ljust(RCRS_LENGTH, "N")) for s in sample_seqs],
    dtype="U1"
)

bases = list("ACGTN")
freqs = np.stack([(seq_matrix == b).mean(axis=0) for b in bases], axis=0)  # (5, 16569)

# Shannon entropy at each position (log2, bits)
pos_entropy = np.array(
    [scipy_entropy(freqs[:, pos] + 1e-9, base=2) for pos in range(RCRS_LENGTH)]
)

# Smoothed for readability (window=50 bp)
window = 50
smoothed = np.convolve(pos_entropy, np.ones(window) / window, mode="same")

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(smoothed, lw=0.8, color="#2c7bb6", label="Entropy (50 bp smoothed)")
ax.axvspan(DLOOP_START, DLOOP_END, alpha=0.12, color="orange", label="D-loop region")
ax.set_xlabel("Genomic position (bp)")
ax.set_ylabel("Shannon entropy (bits)")
ax.set_title("Per-position nucleotide diversity across the mtDNA corpus")
ax.set_xlim(0, RCRS_LENGTH)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "positional_entropy.png", bbox_inches="tight")
plt.show()

dloop_entropy  = pos_entropy[DLOOP_START:DLOOP_END].mean()
coding_entropy = np.concatenate([pos_entropy[:DLOOP_START], pos_entropy[DLOOP_END:]]).mean()
ratio = dloop_entropy / coding_entropy if coding_entropy > 0 else float("inf")
print(f"Mean entropy — D-loop: {dloop_entropy:.4f} bits")
print(f"Mean entropy — coding: {coding_entropy:.4f} bits")
print(f"D-loop / coding ratio: {ratio:.1f}x")

## 3. N-Content Distribution

Sequences with >10% N characters (our QC threshold) are low-quality and likely to harm training.
This plot shows how the threshold was chosen and what fraction of the corpus is excluded.

In [ ]:
n_fracs = df["sequence"].dropna().map(lambda s: s.count("N") / len(s))
threshold = 0.10
n_fail = (n_fracs > threshold).sum()
n_total = len(n_fracs)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(n_fracs, bins=60, color="steelblue", edgecolor="white", log=True)
ax.axvline(threshold, color="firebrick", lw=1.5, linestyle="--",
           label=f"QC threshold ({threshold:.0%})")
ax.set_xlabel("N-content fraction")
ax.set_ylabel("Number of sequences (log scale)")
ax.set_title("N-content distribution before QC filtering")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "n_content_distribution.png", bbox_inches="tight")
plt.show()

print(f"QC fail (>{threshold:.0%} N): {n_fail:,} / {n_total:,} ({n_fail/n_total:.1%})")

## 4. Sequence Length Distribution Before Normalization

All output sequences must be exactly 16,569 bp (rCRS length).
Sequences shorter than this get padded at the D-loop start; longer ones are trimmed.
This plot shows how much adjustment is needed — and whether the raw data is already close.

In [ ]:
lengths = df["length_raw"].dropna()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(lengths, bins=80, color="steelblue", edgecolor="white")
ax.axvline(RCRS_LENGTH, color="firebrick", lw=1.5, linestyle="--",
           label=f"rCRS length ({RCRS_LENGTH:,} bp)")
ax.set_xlabel("Sequence length (bp)")
ax.set_ylabel("Number of sequences")
ax.set_title("Raw sequence length distribution (before normalization)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "length_distribution.png", bbox_inches="tight")
plt.show()

n_short = (lengths < RCRS_LENGTH).sum()
n_long  = (lengths > RCRS_LENGTH).sum()
n_exact = (lengths == RCRS_LENGTH).sum()
print(f"Shorter than rCRS: {n_short:,} ({n_short/len(lengths):.1%})")
print(f"Exact rCRS length: {n_exact:,} ({n_exact/len(lengths):.1%})")
print(f"Longer than rCRS:  {n_long:,}  ({n_long/len(lengths):.1%})")

## 5. Geographic Distribution of HmtDB Samples

HmtDB over-represents European sequences, which means H-clade haplogroups dominate the training corpus.
This is a known limitation. The model may perform worse on under-represented haplogroups (L0, B, C, D)
than on European ones.

In [ ]:
geo_col = "geographic_origin"
if geo_col in df.columns and df[geo_col].notna().sum() > 0:
    geo_counts = df[geo_col].dropna().value_counts().head(15)

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(geo_counts.index[::-1], geo_counts.values[::-1],
                   color="steelblue", edgecolor="white")
    ax.set_xlabel("Number of sequences")
    ax.set_title("Geographic distribution of HmtDB samples (top 15 regions)")
    ax.bar_label(bars, fmt="%d", fontsize=8, padding=3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "geographic_distribution.png", bbox_inches="tight")
    plt.show()

    european = df[geo_col].dropna().str.contains(
        "Europe|European|Italy|Germany|Spain|UK|France", case=False, regex=True
    ).sum()
    total = df[geo_col].notna().sum()
    print(f"European samples: {european:,} / {total:,} ({european/total:.1%})")
else:
    print("geographic_origin column not available in this dataset.")

## Summary

| Metric | Value |
|--------|-------|
| Total sequences | — |
| QC pass rate | — |
| D-loop entropy / coding entropy | ~7x |
| H-haplogroup fraction | ~55% (European bias) |

The D-loop entropy contrast confirms that circular positional encoding is necessary:
the model must learn that positions 16,024–16,569 and 1–576 form a contiguous high-entropy region,
not two separate ones. Standard sinusoidal PE treats them as 16,568 positions apart.